# 17 · What it is allowed to do, and what it is not

The agents can now diagnose. The dangerous idea is to let them fix.

This notebook is about the boundary: what the system produces, what it refuses
to do, and where a person is put in the way on purpose.

![](img/oncall-4-artifacts.png)

> **It proposes. A person decides. Nothing it does changes a number.**

In [ ]:
import sys; sys.path.insert(0, '..')
from pipelines.lib.config import dsn, SCHEMA

---

## Four artifacts, and one of them is not optional

| Tool | Produces | When |
|---|---|---|
| `publish_incident_page` | a Confluence page with a diagram | **always** |
| `raise_ticket` | a tracked item with an owner | **always** |
| `propose_code_change` | a branch and a pull request | when the fix is code |
| `request_db_change` | an approval request. Runs nothing | when the fix touches data |

### Why a page is always produced

Because the most common outcome of an investigation is not a fix. It is a
diagnosis, some evidence, and a decision that belongs to somebody who was
asleep.

**A system that only produces output when it is confident produces nothing on
the nights you needed it most.**

---

## The page

Composed in code, not by the model. The agent supplies eight strings; the shape,
the ordering, the macros and the column widths are fixed here.

In [ ]:
from agent_service.tools import confluence as cf

print('the pieces a page is built from:\n')
for name in ('details', 'status', 'toc', 'panel', 'code', 'expand', 'table', 'image'):
    fn = getattr(cf, name)
    print(f'  {name:10} {(fn.__doc__ or "").strip().splitlines()[0][:66]}')

### Why the model does not write the XHTML

Ask a model for Confluence storage format and you get a different page every
time, some of them broken. Ask it for eight strings and assemble them yourself
and **every page in the space looks the same**, which is what makes a space
navigable rather than a pile.

The structure, in order: a summary table, a provenance panel, a table of
contents, what happened, a diagram, evidence in code blocks, what it means,
numbered steps, an ownership table, and a collapsed appendix.

**A reader should be able to stop after the summary table and still know what to
do.** Everything below the fold is for the person who wants to check the work.

In [ ]:
summary = cf.details([
    ('Status',   cf.status('open')),
    ('Severity', cf.status('high')),
    ('Owner',    '<p>pricing</p>'),
])
print(summary[:400], '...')

That is the page properties macro. Confluence can roll those up into a report,
so a parent page becomes an index of every incident **without anybody
maintaining one**.

## Is Confluence configured here?

In [ ]:
print('configured:', cf.configured())
if not cf.configured():
    print('\nNo Atlassian credentials, so pages are written to artifacts/ as markdown.')
    print('The whole project still runs. Set ATLASSIAN_SITE, ATLASSIAN_EMAIL,')
    print('ATLASSIAN_TOKEN and CONFLUENCE_SPACE in .env to publish for real.')
else:
    print('site  :', cf.SITE)
    print('space :', cf.SPACE_KEY)
    print('parent:', cf.PARENT_TITLE)

### The seam

`Publisher` is an interface with two implementations: markdown on disk, or the
Confluence REST API. It turns itself on when the credentials are present.

**Swapping one for the other changes no agent code.** That is the entire reason
the interface exists, and it is why this project works for a student with no
Atlassian account and for you with one.

---

## The diagram

An incident page should answer the question everybody asks first and nobody
writes down: **where in the flow did this break?**

In [ ]:
from agent_service.tools import diagram

png = diagram.render(
    breach_id='DEMO',
    kpi='surge_coverage_pct',
    title='Surge stopped arriving for one release',
    broken_at='the contract in pipelines/p3_bronze_driver_app.py',
    what_broke='The loader knows four paths for surge. The release sends a fifth.')
print('drawn:', png)

In [ ]:
from IPython.display import Image, display
if png:
    display(Image(str(png), width=980))

Green is proved fine. Red is where the value stops being readable. **Amber is
every layer after it, which keeps working perfectly and reports a number built
on less data than it should be.**

Nothing fails. No row count changes. That is the whole problem, drawn.

---

## Code changes: a branch and a pull request, never a merge

In [ ]:
from agent_service.tools.publish import WRITABLE, propose_code_change
print('directories a proposed change may touch:', WRITABLE)
print()
print(propose_code_change.invoke({
    'breach_id': 'DEMO', 'summary': 'x', 'rationale': 'x',
    'path': 'platform/docker-compose.yml',
    'old_string': 'a', 'new_string': 'b'}))

**An agent that can edit the tests which judge it is not being judged.**

And before a human is even asked, the edit is applied in memory and the result
is parsed:

In [ ]:
print(propose_code_change.invoke({
    'breach_id': 'DEMO',
    'summary': 'deliberately break the syntax',
    'rationale': 'to show the gate',
    'path': 'signals/board.py',
    'old_string': 'Z_THRESHOLD = 4.0',
    'new_string': 'Z_THRESHOLD = = 4.0'}))

> **Whether a change is right is a judgement. Whether it compiles is a fact.**
> Facts get checked before a person is asked for an opinion.

---

## Data changes: the graph stops

This is the one the user is put in the way of, deliberately.

In [ ]:
from agent_service.tools.publish import request_db_change
print(request_db_change.invoke({
    'breach_id': 'DEMO',
    'summary': 'Backfill surge for release 4.2.0',
    'rationale': 'The value exists in the source at a path the contract now reads.',
    'statement': "UPDATE teach.bronze_driver_app SET surge = surge_backup "
                 "WHERE surge IS NULL AND app_version = '4.2.0';",
    'rows_affected_estimate': '395 rows, counted with a SELECT before proposing this',
    'reversible': 'Yes: surge_backup is untouched, so setting surge back to NULL restores it.',
}))

### Read what it did not do

It did not run the statement. **No part of this system will run it.** A person
does that, by hand, having read it.

And it insisted on three things before it would even record the request: the
exact statement, an estimate of the blast radius, and **how to undo it**. Never
propose a data change without saying how to reverse it.

## And the graph itself pauses

In [ ]:
from agent_service.tools.publish import GATED
from agent_service.supervisor import build_supervisor
import inspect

print('tools that stop for a human:', GATED)
print()
print(inspect.getsource(build_supervisor))

`HumanInTheLoopMiddleware` pauses the graph **before** a gated tool runs. The
investigation comes back with `waiting_for_human: True` and nothing has
happened yet.

Resuming takes a decision and a reason:

```python
resume(thread_id, breach, approve=False, note="we are rolling the app back instead")
```

**Rejecting is a real outcome and the reason goes back to the agent.** That is
the difference between a human gate and a rubber stamp.

### Why code changes do not pause here

A pull request is already a human gate, by construction. Pausing twice for the
same decision teaches people to click through, and a gate everybody clicks
through is worse than no gate, because it looks like control.

---

## What you learned

- **A page and a ticket are always produced**, even when the diagnosis is uncertain
- The page is **composed in code**, so every page in the space looks the same
- The publisher is **an interface**, so no credentials still works
- Code changes go to **a branch and a pull request**, never a merge, and only in
  two directories
- The edit is **parsed before a human is asked**, because compiling is a fact
- Data changes **stop the graph**. Nothing is executed, ever
- A proposed data change must say **how to reverse it**